# Tema 12 · Bloque 1 (Jueves) — Frameworks y configuración del entorno
### Los 4 retos, en corto

**El tema en 4 líneas:**
1. Un **framework** (Qiskit, Cirq, Braket) es el puente entre las matemáticas y el código ejecutable.
2. **Aer** es el simulador local de Qiskit: rápido, gratis, sin internet ni token.
3. Un **entorno virtual** es una caja de herramientas por proyecto, para que no se rompan entre sí.
4. El **token** es tu contraseña: nunca va escrito en el código.

> Este tema no tiene física. Es puro "cómo montar el taller antes de trabajar".


## Instalación (ejecuta esto primero)

In [ ]:
try:
    import qiskit, qiskit_aer
except ModuleNotFoundError:
    import subprocess, sys
    print("Instalando...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qiskit", "qiskit-aer"], check=True)
    import qiskit, qiskit_aer

print("Qiskit", qiskit.__version__, "· Aer", qiskit_aer.__version__)


---
# RETO 1 — Simulador local Aer

**Qué es Aer:** el simulador de Qiskit que corre **en tu máquina**. Imita una computadora cuántica sin ruido.

**Qué hace el circuito:** un qubit, una compuerta `H` (superposición), y lo mide 500 veces.
Como está en superposición, debe salir ~50% `0` y ~50% `1`.


In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def verificar_simulacion_local():
    qc = QuantumCircuit(1, 1)   # 1 qubit, 1 bit clásico para guardar la medición
    qc.h(0)                     # H = superposición: |0> y |1> a la vez
    qc.measure(0, 0)            # medir el qubit 0 y guardarlo en el bit 0
    sim = AerSimulator()        # el simulador, en tu máquina
    resultado = sim.run(qc, shots=500).result()   # ejecutar 500 veces
    return resultado.get_counts()

print("Resultados del simulador local:", verificar_simulacion_local())


In [ ]:
# ¿Cuánto tarda? Comparemos con lo que tardaría ir a la nube.
import time

t0 = time.perf_counter()
verificar_simulacion_local()
local_ms = (time.perf_counter() - t0) * 1000

print(f"Simulador local : {local_ms:>8.1f} ms")
print(f"QPU en la nube  : {'minutos a horas':>15}  (cola compartida + red)")
print("\nLa diferencia no es la velocidad del cálculo: es que NO SALE de tu computadora.")


### ✅ Respuesta Reto 1

**¿Por qué Aer no tiene latencia de red ni pide token?**

Porque **todo ocurre dentro de tu computadora**. Aer no es hardware cuántico: es un programa en C++ que
calcula la física con matemáticas clásicas.

- **Sin red** → no hay peticiones HTTP, no hay cola, no hay servidor. Solo tu CPU y tu RAM.
- **Sin token** → el token existe para controlar *quién usa las QPU de IBM*, que son un recurso caro y
  compartido. Tu propia CPU no necesita permiso de nadie.

**El matiz:** Aer es gratis pero tiene techo. Simular `n` qubits necesita `2ⁿ` amplitudes en RAM, así que
alrededor de **30 qubits ya no cabe**. Ahí sí toca la QPU real.


---
# RETO 2 — Qiskit vs Cirq vs Braket

Los tres hacen lo mismo (circuitos cuánticos). Cambian el **dueño** y el **enfoque**.


In [ ]:
frameworks = [
    ("Qiskit",  "IBM",    "Pedagógico y abierto",     "QPUs de IBM",              "Suave",      "Aprender y prototipar"),
    ("Cirq",    "Google", "Control fino del hardware","Google Sycamore",          "Pronunciada","Diseño NISQ avanzado"),
    ("Braket",  "AWS",    "Interfaz multivendor",     "IonQ, Rigetti, IQM, AQT",  "Media",      "Empresas ya en AWS"),
]

print(f"{'SDK':<9}{'Dueño':<9}{'Filosofía':<28}{'Hardware':<28}{'Curva':<13}Caso ideal")
print("-" * 118)
for f in frameworks:
    print(f"{f[0]:<9}{f[1]:<9}{f[2]:<28}{f[3]:<28}{f[4]:<13}{f[5]}")


In [ ]:
# Mini guía de decisión
def recomendar(objetivo):
    reglas = {
        "aprender":      "Qiskit  — comunidad enorme, documentación y tutoriales",
        "google":        "Cirq    — control de la topología de Sycamore",
        "multivendor":   "Braket  — varios proveedores con una sola API",
        "aws":           "Braket  — integración nativa con la infraestructura AWS",
        "portabilidad":  "Qiskit  — estándar de facto, con puentes hacia otros SDKs",
    }
    return reglas.get(objetivo, "Depende: primero define la restricción principal")

for obj in ["aprender", "google", "multivendor", "aws", "portabilidad"]:
    print(f"  Si tu objetivo es '{obj}':".ljust(38), recomendar(obj))


### ✅ Respuesta Reto 2

**¿Qué framework para investigación algorítmica general con alta portabilidad entre superconductores e
iones atrapados?**

Hay dos lecturas válidas, y conviene decir las dos:

- **Si lo que pesa es ejecutar en hardware de varios fabricantes → Amazon Braket.** Es el único que da acceso
  a superconductores (Rigetti, IQM) **e** iones atrapados (IonQ) desde **una sola API**, sin reescribir el
  proyecto al cambiar de proveedor. Portabilidad de *ejecución*.

- **Si lo que pesa es la investigación algorítmica en sí → Qiskit.** Es el estándar de facto: la mayoría de
  papers, tutoriales y librerías (Qiskit Nature, nuevos algoritmos) salen primero ahí, y existen puentes para
  exportar circuitos a otros SDKs. Portabilidad de *conocimiento y código*.

**Respuesta equilibrada:** desarrollar e investigar en **Qiskit**, y usar **Braket** como capa de ejecución
cuando haga falta comparar hardware de distintos fabricantes.

**Por qué no Cirq aquí:** su fuerza es el control fino de la topología de Sycamore (Google) — justo lo
contrario de la portabilidad entre fabricantes.


---
# RETO 3 — Entornos virtuales

**El problema:** Python instala las librerías en **un solo lugar compartido**. Si el Proyecto A necesita
NumPy 1.24 y el Proyecto B necesita NumPy 2.0, uno de los dos se rompe.

**La solución:** un entorno virtual = una carpeta con su propia copia de las librerías. Una caja por proyecto.


In [ ]:
import sys, numpy, scipy, qiskit

print("MI ENTORNO ACTUAL")
print("-" * 55)
print("Python  :", sys.version.split()[0])
print("NumPy   :", numpy.__version__)
print("SciPy   :", scipy.__version__)
print("Qiskit  :", qiskit.__version__)

en_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
print("\n¿Estoy dentro de un entorno virtual?:", "SÍ" if en_venv else "NO (entorno global o Colab)")
print("Ubicación de las librerías:", sys.prefix)


In [ ]:
# El choque de versiones, ilustrado
proyectos = [
    ("Proyecto A (cuántico)",    "qiskit-aer 0.15", "numpy >=1.24, <2.0"),
    ("Proyecto B (machine learning)", "tensorflow 2.17", "numpy >=2.0"),
]
print(f"{'Proyecto':<32}{'Librería principal':<20}{'Necesita'}")
print("-" * 76)
for p in proyectos:
    print(f"{p[0]:<32}{p[1]:<20}{p[2]}")

print("\nEn un Python GLOBAL solo puede haber UNA versión de numpy instalada.")
print("-> Instalas el Proyecto B y el Proyecto A se rompe, SIN avisar.")
print("-> Eso es una 'actualización silenciosa'.\n")
print("Con entornos virtuales: cada proyecto tiene SU numpy. Ninguno se entera del otro.")


### Los comandos (para tu máquina, no para Colab)

```bash
python3 -m venv entorno_cuantico          # 1. crear la caja
source entorno_cuantico/bin/activate      # 2. entrar (Linux/Mac)
entorno_cuantico\Scripts\activate         #    (Windows)
pip install qiskit qiskit-aer             # 3. instalar SOLO aquí dentro
deactivate                                # 4. salir
```

### ✅ Respuesta Reto 3

**¿Cómo previene un venv el fallo de compatibilidad de binarios C/Fortran?**

Simuladores como Aer no son Python puro: son **binarios compilados en C++** que se enlazan contra una versión
**concreta** de NumPy (su ABI, la forma exacta en que los datos viven en memoria).

Si actualizas NumPy por debajo, ese binario sigue esperando la estructura vieja → `ImportError`, resultados
corruptos o crash. Y **no se detecta al instalar**: falla después, en ejecución.

El venv lo evita porque **congela el conjunto completo de versiones en una carpeta propia**. Instalar algo en
otro proyecto no toca esta carpeta, así que el binario compilado siempre encuentra exactamente la versión
contra la que fue enlazado.

**venv vs Conda:** `venv` aísla **paquetes de Python**. Conda además aísla **librerías del sistema**
(compiladores, BLAS/LAPACK, CUDA). Para simuladores pesados, Conda suele ser más seguro.


---
# RETO 4 — Seguridad del token

**El token de IBM Quantum es tu contraseña.** Si alguien lo tiene, puede consumir tu tiempo de QPU y actuar
en tu nombre.

**La regla de oro:** el token **nunca** se escribe en el código.


In [ ]:
# MAL vs BIEN
print("MAL  -> token escrito en el código (queda en Git para siempre):")
print('       QiskitRuntimeService(token="a1b2c3d4e5f6...")\n')

print("BIEN -> opción 1: guardarlo UNA vez en el almacén seguro del SDK")
print('       QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...")')
print('       QiskitRuntimeService()      # las siguientes veces, lo lee solo\n')

print("BIEN -> opción 2: variable de entorno (ideal para servidores y CI)")
import os
token = os.environ.get("IBM_QUANTUM_TOKEN")
print("       token = os.environ.get('IBM_QUANTUM_TOKEN')")
print("       Valor leído ahora mismo:", "encontrado" if token else "no configurado (normal aquí)")


In [ ]:
# Detector simple de secretos: lo que hace un escáner de seguridad en un repo
import re

codigo_ejemplo = '''
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService(token="8f14e45fceea167a5a36dedd4bea2543abc123def456")
API_KEY = "sk-proj-9f8e7d6c5b4a3210"
password = os.environ.get("DB_PASSWORD")
'''

patrones = [
    (r'token\s*=\s*["\'][A-Za-z0-9_\-]{20,}["\']', "Token hardcodeado"),
    (r'API_KEY\s*=\s*["\'][A-Za-z0-9_\-]{10,}["\']', "API key hardcodeada"),
]

print("ESCANEO DEL CÓDIGO\n")
hallazgos = 0
for linea in codigo_ejemplo.strip().split("\n"):
    for patron, tipo in patrones:
        if re.search(patron, linea):
            hallazgos += 1
            print(f"  [!] {tipo}: {linea.strip()[:60]}...")

print(f"\n{hallazgos} secretos expuestos encontrados.")
print("La línea con os.environ.get() NO se marca: ahí no hay ningún secreto escrito.")


### Protección básica: `.gitignore`

```
.env
*.key
config/secrets.json
```

Así Git ignora esos archivos y nunca se suben.

### ✅ Respuesta Reto 4

**¿Qué hacen las plataformas cloud cuando se filtra una clave en un commit público?**

**1. Detección automática (*secret scanning*).** GitHub, GitLab y AWS escanean **cada push** buscando patrones
de credenciales conocidas. Los proveedores registran el formato de sus tokens para que sean reconocibles.

**2. Revocación inmediata.** Al detectarla, la plataforma **invalida el token al instante** y notifica al dueño
por email. La clave deja de funcionar — para el atacante y para ti.

**3. Auditoría.** Los logs registran cada uso de la credencial (cuándo, desde qué IP, qué trabajos envió), lo
que permite ver si alguien la usó antes de la revocación.

**4. Rotación.** Se genera un token nuevo y se reemplaza donde hiciera falta.

**Lo crítico:** borrar el commit **no sirve**. Git guarda todo el historial, y los bots que rastrean GitHub
encuentran claves nuevas **en segundos**. Una vez publicada, esa clave está quemada para siempre: la única
salida real es **revocarla y generar otra**.

**Por qué es tan grave con IBM Quantum:** el tiempo de QPU es un recurso escaso y con cola. Un token filtrado
significa que otro puede gastar tu cuota, colarse con tus prioridades y ejecutar trabajos a tu nombre.


---
## Resumen del tema

| Tema | Idea |
|---|---|
| **Framework** | El puente entre las matemáticas y el código ejecutable |
| **Aer** | Simulador local: sin red, sin token, sin cola — pero con techo (~30 qubits) |
| **Qiskit / Cirq / Braket** | Aprender / control de Google / multivendor en AWS |
| **Entorno virtual** | Una caja de librerías por proyecto: evita que uno rompa a otro |
| **Token** | Contraseña personal: va en el almacén del SDK o en variable de entorno, **nunca** en el código |

**Preguntas de cierre del PDF:**
- *¿Cómo influye la elección del framework?* Determina a qué hardware llegas, cuánto tarda el equipo en ser
  productivo y cuánto costaría cambiar de proveedor después. Elegir mal no impide trabajar, pero encarece
  cada paso siguiente.
- *¿Riesgos de omitir entornos virtuales?* Dependencias que se rompen en silencio, resultados no reproducibles
  ("en mi máquina sí funciona") y despliegues que fallan porque nadie sabe qué versiones hacían falta.
